# 02 — Blocking analysis
Blocking recall is the ceiling for everything downstream. This notebook measures recall per
search view, recall vs k, and shows the true pairs blocking misses.

In [ ]:
# --- setup: run from the repo root or from notebooks/ ---
import sys, os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "dataset"                 # real data: dataset/train, dataset/test
if not (DATA / "train").exists():       # fall back to the synthetic dev data
    DATA = ROOT / "fake_dataset"
CACHE = ROOT / "cache"
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("data:", DATA)

In [ ]:
from data import load_split, load_ground_truth, subsample
from blocking import DENSE_VIEWS, SPARSE_VIEWS, generate_candidates, blocking_recall, _sparse_search_mats
from search import encode, topk, sparse_topk, backend_name
FRAC = 0.3          # fraction of train S1 to analyse (keep small for speed)
truth = load_ground_truth(DATA)
s1, s23 = load_split(DATA, "train")
s1, s23 = subsample(s1, s23, truth, FRAC)
id1, id2 = s1["entity_id"].values, s23["entity_id"].values
true_pairs = {(a, b) for a in id1 for b in truth.get(a, ())}
print(f"S1={len(s1)} S23={len(s23)} true links={len(true_pairs)} backend={backend_name()}")

## Recall per view, and what each view adds on its own

In [ ]:
K_MAX = 100
view_hits, view_idx = {}, {}
for v, (col, k, rk) in DENSE_VIEWS.items():
    E1, E2 = encode("svd", pd.concat([s1[col], s23[col]]).values, [s1[col].values, s23[col].values])
    idx, _ = topk(E1, E2, K_MAX)
    view_idx[v] = idx
for v, (col, k, rk) in SPARSE_VIEWS.items():
    A, B = _sparse_search_mats(s1, s23, col)
    idx, _ = sparse_topk(A, B, K_MAX)
    view_idx[v] = idx

def found(idx, k):
    return {(id1[r], id2[c]) for r in range(len(idx)) for c in idx[r, :k] if c >= 0}

default_k = {**{v: k for v, (_, k, _) in DENSE_VIEWS.items()}, **{v: k for v, (_, k, _) in SPARSE_VIEWS.items()}}
sets = {v: found(view_idx[v], default_k[v]) & true_pairs for v in view_idx}
union = set().union(*sets.values())
rows = []
for v, s in sets.items():
    others = set().union(*[x for w, x in sets.items() if w != v])
    rows.append(dict(view=v, k=default_k[v], recall=len(s) / len(true_pairs), unique=len(s - others) / len(true_pairs)))
display(pd.DataFrame(rows))
print(f"union of forward views: {len(union) / len(true_pairs):.4f}")

## Recall vs k (choose k per view here)

In [ ]:
ks = [1, 2, 3, 5, 10, 20, 30, 50, 100]
for v, idx in view_idx.items():
    plt.plot(ks, [len(found(idx, k) & true_pairs) / len(true_pairs) for k in ks], marker="o", label=v)
plt.xscale("log"); plt.xlabel("k"); plt.ylabel("recall"); plt.grid(alpha=.3); plt.legend(); plt.show()

## Full blocking (both directions, all views) as used by the pipeline

In [ ]:
pairs, colstats = generate_candidates(s1, s23)
rec, per, tot = blocking_recall(pairs, s1, s23, truth, id1)
print(f"recall={rec:.4f}  candidates/S1={per:.1f}")
pairs["cnt"] = 1
pairs.groupby("i1")["cnt"].sum().plot.hist(bins=40, title="candidates per S1"); plt.show()

In [ ]:
# recall by country and by source
have = set(zip(id1[pairs.i1], id2[pairs.i2]))
tp = pd.DataFrame(list(true_pairs), columns=["s1", "m"])
tp["hit"] = [(a, b) in have for a, b in zip(tp.s1, tp.m)]
tp["country"] = tp.s1.map(dict(zip(s1.entity_id, s1.country)))
tp["src"] = tp.m.str[:2]
display(tp.groupby("country")["hit"].agg(["mean", "size"]))
display(tp.groupby("src")["hit"].agg(["mean", "size"]))

## The true pairs blocking misses — read these and fix normalisation / add a view

In [ ]:
S1 = s1.set_index("entity_id"); S23 = s23.set_index("entity_id")
miss = tp[~tp.hit]
print(len(miss), "missed")
pd.DataFrame([dict(s1_name=S1.loc[a, "business_name"], s23_name=S23.loc[b, "business_name"],
                   s1_addr=S1.loc[a, "business_address"], s23_addr=S23.loc[b, "business_address"])
              for a, b in zip(miss.s1, miss.m)][:40])